# Basic 03 - Plot and Export
Small, step-by-step plotting/export flow with explicit file-load metadata.


## 1) Imports


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display


## 2) Import wrapper


In [ ]:
from isa_phm import ISAWrapper


## 3) Pick ISA JSON


In [ ]:
# Set this to your dataset's ISA-PHM JSON file.
ISA_JSON = Path("path/to/your-dataset.isa-phm.json").expanduser()
DATA_ROOT = ISA_JSON.parent

print("ISA-JSON :", ISA_JSON)
print("DATA_ROOT:", DATA_ROOT)
print("Exists   :", ISA_JSON.exists())


## 4) Build wrapper (performance knobs shown explicitly)


In [ ]:
wrapper = ISAWrapper(
    ISA_JSON,
    data_root=DATA_ROOT,
    strict_validation=False,
    enable_chunked_large_file_mode=True,
    large_file_threshold_mb=64.0,
    chunk_rows=250_000,
)


## 5) Pick first study


In [ ]:
study = wrapper.study(wrapper.list_studies()[0].title)
study.title


## 6) Pick first assay


In [ ]:
assay = study.assay(study.list_assays()[0].assay_id)
assay.assay_id


## 7) Pick first run


In [ ]:
run_id = assay.list_runs()[0].run_id
run_id


## 8) Load one run with file metadata


In [ ]:
df, meta = assay.load_dataframe_with_meta(run_id=run_id, file_type="auto")


## 9) Show resolved load metadata


In [ ]:
display(pd.DataFrame([meta.model_dump()]))


## 10) Quick data preview


In [ ]:
display(df.head(10))


## 11) Timeseries plot


In [ ]:
fig_ts = assay.plot_timeseries(run_id=run_id, file_type="auto")
fig_ts


## 12) Frequency-domain plot


In [ ]:
fig_fft = assay.plot_frequency_domain(run_id=run_id, file_type="auto")
fig_fft


## 13) Lifecycle features for this assay


In [ ]:
lc = assay.lifecycle_features(file_type="auto")
display(lc.head(10))


## 14) Multi-sensor merged dataframe


In [ ]:
multi_df = study.load_multi_sensor_dataframe(file_type="auto")
display(multi_df.head(10))


## 15) Labeled export dataframe


In [ ]:
labeled_df = study.export_labeled_dataset(file_type="auto")


## 16) Labeled export preview


In [ ]:
display(labeled_df.head(10))
display(pd.DataFrame([{"rows": int(labeled_df.shape[0]), "cols": int(labeled_df.shape[1])}]))


## 17) Validation summary before downstream export


In [ ]:
validation = wrapper.validate_dataset(check_files=True)
display(pd.DataFrame([{
    "ok": validation.ok,
    "n_errors": validation.n_errors,
    "n_warnings": validation.n_warnings,
    "n_info": validation.n_info,
}]))


## 18) AI context export (metadata-only)


In [ ]:
ai_ctx = wrapper.ai_context(include_semantics=True, include_validation=True)
list(ai_ctx.keys())
